# Gold: baseline classifier training

Trains a baseline classifier (ResNet-18) against a published Gold shard export (`notebooks/31_export_gold_shards.ipynb`) using a versioned preprocessing recipe -- identically whether this notebook runs on Databricks or the same logic runs from a local machine via `python -m ml.train` (`src/ml/train.py`'s `run_training` is the one orchestrator both paths call). Logs hyperparameters, metrics, the model artifact and the run's inputs (the Gold manifest release and the shard files it read) to MLflow. If the training-run config sets `registered_model_name` (e.g. `derm_showcase_project.gold.baseline_classifier`), the trained model is also registered as a new version of that Unity Catalog model -- so training a different architecture or hyperparameter set later accumulates as a new version of the same named model instead of only being addressable by MLflow `run_id`. See `docs/decisions/007-pin-data-and-preprocessing-per-training-run.md` and `docs/decisions/008-baseline-model-resnet18.md` for the full design.

**Prerequisites**: `30_create_gold_manifest.ipynb` and `31_export_gold_shards.ipynb` must already have run for the target `dataset_version`. This notebook's serverless Environment needs `torch`, `torchvision`, `mlflow-skinny`, and `scikit-learn` added as dependencies (same mechanism already set up for `mosaicml-streaming` on notebook `31` -- `torch`/`torchvision` may already be present transitively via that Environment entry, confirm before running). A local run additionally needs `DATABRICKS_HOST`/`DATABRICKS_TOKEN` set (or a configured `~/.databrickscfg` profile) so MLflow can reach the Databricks-hosted tracking server.

In [ ]:
%run ./_setup_env


In [ ]:
import mlflow

from data_platform.dataset_layout import load_yaml_config
from ml.train import resolve_shards_root, run_training
from ml.training_run import resolve_training_run

# Which training run to build -- one small YAML file per run under config/gold/training_runs/,
# mirroring config/gold/manifests/<name>.yaml's convention. To try a different
# dataset_version/preprocessing_version/hyperparameter combination, add a new YAML file there
# rather than editing values in place here.
TRAINING_RUN_NAME = "sample-v1-resnet18"

print(f"Training run: {TRAINING_RUN_NAME}")


## Resolve Training Run

`resolve_training_run` loads `config/gold/training_runs/{TRAINING_RUN_NAME}.yaml`, validates it, then loads and validates its referenced preprocessing config -- the one place a run name turns into everything training needs. Raises `ValueError` on any missing/malformed field in either config.


In [ ]:
TRAINING_RUN_SPEC = resolve_training_run(CONFIG_ROOT, TRAINING_RUN_NAME)

print(f"dataset_version: {TRAINING_RUN_SPEC.dataset_version}")
print(f"preprocessing_version: {TRAINING_RUN_SPEC.preprocessing_version}")
print(f"architecture: {TRAINING_RUN_SPEC.architecture}, pretrained: {TRAINING_RUN_SPEC.pretrained}")
print(f"batch_size: {TRAINING_RUN_SPEC.batch_size}, num_epochs: {TRAINING_RUN_SPEC.num_epochs}, learning_rate: {TRAINING_RUN_SPEC.learning_rate}")
print(f"label_values: {TRAINING_RUN_SPEC.label_values}")


## Resolve Shard Root

On Databricks this resolves for free -- the Gold shard export's Volume path is directly mounted, no copy needed. On a local machine, pass `--shards-root` to `python -m ml.train` pointing at a `databricks fs cp -r`'d copy of the shard directory instead (see `docs/data_contract.md`'s Gold shard export section) -- this notebook cell always resolves the Databricks-mounted path.


In [ ]:
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
SHARDS_ROOT = resolve_shards_root(None, STORAGE_CONFIG, TRAINING_RUN_SPEC.dataset_version)

print(f"Shards root: {SHARDS_ROOT}")


## Train

`run_training` does everything from here: builds train/validation/test dataloaders with the resolved preprocessing transforms, fine-tunes the baseline model, logs hyperparameters/per-epoch metrics/final test metrics (balanced accuracy, per-class recall, confusion matrix, classification report)/the model artifact to MLflow, and returns the MLflow `run_id`. Nothing training-specific lives in this notebook -- it is all in `src/ml/train.py`, identically callable from a local script.


In [ ]:
RUN_ID = run_training(
    TRAINING_RUN_NAME,
    config_root=CONFIG_ROOT,
    shards_root=SHARDS_ROOT,
)

print(f"MLflow run_id: {RUN_ID}")


## Review Outputs

Pulls this run'''s logged metrics back from MLflow for a quick sanity check, same "review outputs" convention every other Gold-stage notebook ends with.


In [ ]:
run_metrics = mlflow.MlflowClient().get_run(RUN_ID).data.metrics
for metric_name, value in sorted(run_metrics.items()):
    print(f"{metric_name}: {value}")
